<a href="https://colab.research.google.com/github/Raju-kushwaha1230/ML/blob/main/TrainModel_Using_Pipeline_TitanicDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd


In [11]:
df = pd.read_csv('/content/train.csv')

In [12]:
df.sample(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
568,569,0,3,"Doharr, Mr. Tannous",male,NaN,0,0,2686,7.2292,NaN,C
723,724,0,2,"Hodges, Mr. Henry Price",male,50.0,0,0,250643,13.0000,NaN,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
df = df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

In [14]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=102)

In [17]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
190,2,female,32.0,0,0,13.000,S
456,1,male,65.0,0,0,26.550,S
853,1,female,16.0,0,1,39.400,S
374,3,female,3.0,3,1,21.075,S
333,3,male,16.0,2,0,18.000,S


In [53]:
X_test.head(

)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
618,2,female,4.0,2,1,39.0000,S
849,1,female,NaN,1,0,89.1042,C
235,3,female,NaN,0,0,7.5500,S
865,2,female,42.0,0,0,13.0000,S
731,3,male,11.0,0,0,18.7875,C


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

# **Create Transformer**

In [27]:
trf1 = ColumnTransformer([
    ('imute_age',SimpleImputer(),[2]),
    ('imute_embarked', SimpleImputer(strategy='most_frequent'), [6])
], remainder = 'passthrough')

In [62]:
trf2 = ColumnTransformer([
    ('onehot_sex', OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])

],remainder='passthrough')

In [63]:
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(),slice(0,10))
])

In [64]:
# feature selection
from sklearn.feature_selection import SelectKBest, chi2
trf4  = SelectKBest(score_func=chi2, k=8)

In [65]:
# train the model
from sklearn.tree import DecisionTreeClassifier
trf5 = DecisionTreeClassifier()

# **Create Pipeline**
```
# using Pipeline class
```
```
# using make_pipeline
```

In [66]:
from sklearn.pipeline import Pipeline, make_pipeline

In [67]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3',trf3),
    ("trf4", trf4),
    ('trf5', trf5)
])

In [68]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imute_age', SimpleImputer(),
                                                  [2]),
                                                 ('imute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehot_sex',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7c1f98cec900>)),
                ('trf5', DecisionTreeClassifier())])

# **Explore the pipeline**

In [69]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

array([29.68461806])

In [70]:
pipe.named_steps['trf2'].transformers_

[('onehot_sex',
  OneHotEncoder(handle_unknown='ignore', sparse_output=False),
  [1, 6]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 2, 3, 4, 5])]

In [71]:
from sklearn.metrics import accuracy_score

In [72]:
y_pred = pipe.predict(X_test)

In [75]:
accuracy_score(y_test, y_pred)

0.5977653631284916

# **Cross validation using pipeline**

In [77]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train,y_train,cv= 5, scoring='accuracy').mean()

np.float64(0.6460750517088545)

In [78]:
import pickle
pickle.dump(pipe, open('pipe.pkl','wb'))

# **Predict using Trained Model|**

In [79]:
model = pickle.load(open('/content/pipe.pkl','rb'))

In [80]:
test_input = np.array([2,'make',32,0,0,10.5,'S'], dtype=object).reshape(1,7)

In [81]:
model.predict(test_input)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0])